# 05 Challenger models

Purpose: XGBoost, LightGBM and CatBoost on the same features as the scorecard.
Each is tuned with Optuna on the validation sample, calibrated on validation,
and combined in a simple average. Outputs are written by `python -m src.pipeline`.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
T = ROOT / "reports" / "tables"; M = ROOT / "models"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## 1. Tuning results

In [ ]:
tuning = json.loads((M / "tuning.json").read_text())
display(pd.DataFrame({k: {"best_iteration": v["best_iteration"],
                          "validation_logloss": v["best_validation_logloss"]} for k, v in tuning.items()}).T)

## 2. Challengers against the scorecard

In [ ]:
comp = pd.read_csv(T / "model_comparison.csv")
display(comp[comp["model"].str.contains("xgboost|lightgbm|catboost|ensemble|scorecard_with_grade_recalibrated")]
        .pivot(index="model", columns="sample", values="gini").round(4))

Read the table as: does any challenger beat the champion on the out-of-time sample by
at least the agreed margin (0.02 Gini), with no larger calibration gap? The verdict is in
notebook 06.

## 3. Which features drive the LightGBM challenger (SHAP)

In [ ]:
shap_imp = pd.read_csv(T / "shap_importance_lightgbm.csv")
display(shap_imp.round(4))

Compare with the scorecard coefficients in `scorecard_coefficients.csv`. Large differences need a business explanation.